# 1. Data and pitch tendencies

Pulls every Statcast pitch for the pitcher (via `pybaseball`, from Baseball Savant), cleans it, and looks at **what he throws and when**. Everything later builds on this table.

Needs network access to `baseballsavant.mlb.com`. Results are cached in `cache/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))          # so cache/ and reports/ live at the repo root
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pitchscout.workspace import Config, get_pitches, get_clips
from pitchscout import plotting as P

# DEMO=True runs on synthetic data with no network (numbers are NOT real).
DEMO = os.environ.get("PITCHSCOUT_DEMO") == "1"
cfg = Config(pitcher="Mason Miller", start="2024-03-28", end="2025-09-28", n_clips=600, demo=DEMO)
if DEMO:
    print("DEMO MODE: synthetic data, results do not describe a real pitcher")

In [ ]:
df, throws = get_pitches(cfg)
print(f"{cfg.pitcher}: {len(df):,} pitches, throws {throws}, {df.game_pk.nunique()} games")
df[["game_date", "pitch_type", "balls", "strikes", "stand", "release_speed", "description"]].head()

## Pitch mix

In [ ]:
from pitchscout import baseline
tend = baseline.tendencies(df)
colors = P.pitch_colors(tend["mix"].index)
P.hbar(tend["mix"], f"{cfg.pitcher}: pitch mix", colors); plt.show()
df.groupby("pitch_type")["release_speed"].describe()[["count", "mean", "std"]].round(1)

## By count

Each row sums to 100%. This is the classic scouting view: what does he go to when he's ahead or behind?

In [ ]:
P.heatmap(tend["by_count"], "Pitch mix by count (row = balls-strikes)"); plt.show()

## By batter side, and what follows each pitch

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
P.heatmap(tend["by_hand"], "By batter side", ax=ax[0])
P.heatmap(tend["after_prev"], "Next pitch (row = previous pitch in the at-bat)", ax=ax[1])
plt.tight_layout(); plt.show()

## Over time (did his mix change between seasons or after the trade?)

In [ ]:
monthly = pd.crosstab(df.game_date.dt.to_period("M"), df.pitch_type, normalize="index")
ax = monthly.plot(figsize=(10, 3.5), color=[colors[c] for c in monthly.columns], lw=2)
ax.set_ylabel("share"); ax.set_title("Pitch mix by month", loc="left"); plt.show()